# Spark Transformation Prototype
This notebook demonstrates a basic PySpark ETL pipeline: 
- Extraction from Google Cloud Storage
- Transformation
- Export to GCS.

## Environment Variables

Get

In [1]:
import os
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

GC_PROJECT_ID = os.getenv("GCP_PROJECT_ID")
GC_BUCKET = os.getenv("GCS_BUCKET_NAME")
DATA_SUBDIR = os.getenv("DATA_SUBDIR")
RAW_FILE_NAME = os.getenv("RAW_FILE_NAME")

INPUT_DIR = f"gs://{GC_BUCKET}/{DATA_SUBDIR}/raw"
OUTPUT_DIR = f"gs://{GC_BUCKET}/{DATA_SUBDIR}/processed"

# Raw data files should be placed in the GCS bucket
RAW_FILE_PATH = f"{INPUT_DIR}/{RAW_FILE_NAME}"
print(f"Input files path: {RAW_FILE_PATH}")

Input files path: gs://bucket_blent_spark/data/raw/sample.csv


Set

In [2]:
# expanduser() translates the ~ to current user's home directory
credentials_path = os.path.expanduser("~/.config/gcloud/application_default_credentials.json")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = credentials_path

## Initialize Spark Session 
Spark requires a connector to access the large files stored in GCS.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Google Cloud Storage : Spark accesses to large data files in GCP
spark = SparkSession.builder \
    .appName("Spark_Read_50GB_in_GCS") \
    .config(  # 1. Downloads JAR with Spark's driver to access GCS.
            "spark.jars.packages",
            "com.google.cloud.bigdataoss:gcs-connector:hadoop3-2.2.8") \
    .config(  # 2. Adopts the use of "gs://" in paths.
            "spark.hadoop.fs.gs.impl",
            "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
     .config(  # 2. Login via personal App. Default Credentials
          "spark.hadoop.fs.gs.auth.type", "APPLICATION_DEFAULT") \
    .config(  # 3. Provide your Project ID
         "spark.hadoop.fs.gs.project.id", GC_PROJECT_ID) \
    .getOrCreate()  # Gets Spark running session or creates new one

    .config(  # 3. Doesn't login via unavailable Service Account JSON key.
            "spark.hadoop.google.cloud.auth.service.account.enable", 
            "false") \
    .config(  # 4. Login via personal App. Default Credentials
            "spark.hadoop.google.cloud.auth.type", 
            "USER_CREDENTIALS") \

print("\n***** Spark Session Running *****")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/04 17:30:19 WARN Utils: Your hostname, LAPTOP-HP, resolves to a loopback address: 127.0.1.1; using 192.168.1.110 instead (on interface wlo1)
26/06/04 17:30:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/run/media/jean/736060f4-2547-4ca7-aa7d-36044901db71/jean/Documents/Dev/blent/Blent_Spark/venv_spark/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jean/.ivy2.5.2/cache
The jars for the packages stored in: /home/jean/.ivy2.5.2/jars
com.google.cloud.bigdataoss#gcs-connector added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-86369d49-a702-4f9a-8ad0-56c25759f092;1.0
	confs: [default]
	found com.google.cloud.bigdataoss#gcs-connector;hadoop3-2.2.8 in central
	found com.google.api-client#google-api-client-jackson2;1.32.2 i


***** Spark Session Running *****


In [4]:
# 2. Ingestion: Read raw data
df = spark.read.csv(RAW_FILE_PATH, header=True, inferSchema=True)

print(f"Read {df.count()} rows")
display(df.show(5))
display(list(df.schema))

Read 1244245 rows
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|         event_time|event_type|product_id|        category_id|       category_code|   brand|  price|  user_id|        user_session|
+-------------------+----------+----------+-------------------+--------------------+--------+-------+---------+--------------------+
|2019-10-01 00:00:00|      view|  44600062|2103807459595387724|                NULL|shiseido|  35.79|541312140|72d76fde-8bb3-4e0...|
|2019-10-01 00:00:00|      view|   3900821|2053013552326770905|appliances.enviro...|    aqua|   33.2|554748717|9333dfbd-b87a-470...|
|2019-10-01 00:00:01|      view|  17200506|2053013559792632471|furniture.living_...|    NULL|  543.1|519107250|566511c2-e2e3-422...|
|2019-10-01 00:00:01|      view|   1307067|2053013558920217191|  computers.notebook|  lenovo| 251.74|550050854|7c90fc70-0e80-459...|
|2019-10-01 00:00:04|      view|   1004237|20530135

None

[StructField('event_time', TimestampType(), True),
 StructField('event_type', StringType(), True),
 StructField('product_id', IntegerType(), True),
 StructField('category_id', LongType(), True),
 StructField('category_code', StringType(), True),
 StructField('brand', StringType(), True),
 StructField('price', DoubleType(), True),
 StructField('user_id', IntegerType(), True),
 StructField('user_session', StringType(), True)]

In [5]:
# 3. Transformation: Filter for 'view' events and select columns
transformed_df = df.filter(col("event_type") == "view") \
    .select("event_time", "brand", "price", "user_id") \
    .limit(1000) # Limit for prototype validation

print("Transformation Complete")
display(transformed_df.show(5))
display(list(transformed_df.schema))

Transformation Complete
+-------------------+--------+-------+---------+
|         event_time|   brand|  price|  user_id|
+-------------------+--------+-------+---------+
|2019-10-01 00:00:00|shiseido|  35.79|541312140|
|2019-10-01 00:00:00|    aqua|   33.2|554748717|
|2019-10-01 00:00:01|    NULL|  543.1|519107250|
|2019-10-01 00:00:01|  lenovo| 251.74|550050854|
|2019-10-01 00:00:04|   apple|1081.98|535871217|
+-------------------+--------+-------+---------+
only showing top 5 rows


None

[StructField('event_time', TimestampType(), True),
 StructField('brand', StringType(), True),
 StructField('price', DoubleType(), True),
 StructField('user_id', IntegerType(), True)]

In [6]:
# 4. Export: Write to processed data folder
#from importlib.resources import path
from gcsfs import GCSFileSystem

output_sub_dir = f"{OUTPUT_DIR}/prototype_output"
print(f"Writing output to {output_sub_dir}/ ...")

#transformed_df.write.mode("overwrite").csv(output_sub_dir, header=True)

success_file = f"{output_sub_dir}/_SUCCESS"
print(f"Checking for success file at {success_file} ...")

if GCSFileSystem().exists(success_file):
    print("Data Upload was successful!")
else:
    print("Data Upload may have failed. Check GCS bucket for output files.")

Writing output to gs://bucket_blent_spark/data/processed/prototype_output/ ...
Checking for success file at gs://bucket_blent_spark/data/processed/prototype_output/_SUCCESS ...


/home/jean/Documents/Dev/blent/Blent_Spark/venv_spark/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Data Upload was successful!
